In [1]:
# 1. imports and paths

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

base = "D:/TNBC_SV_DNA_Repair"
data_dir = os.path.join(base, "dataset")
results_dir = os.path.join(base, "results")
figures_dir = os.path.join(results_dir, "figures")
tables_dir = os.path.join(results_dir, "tables")
scores_dir = os.path.join(results_dir, "scores")

for folder in [results_dir, figures_dir, tables_dir, scores_dir]:
    os.makedirs(folder, exist_ok=True)

print("folders ready")
print("data:", data_dir)
print("results:", results_dir)

folders ready
data: D:/TNBC_SV_DNA_Repair\dataset
results: D:/TNBC_SV_DNA_Repair\results


In [2]:
# 2. gpu check

import torch

print("cuda available:", torch.cuda.is_available())
print("gpu name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("gpu memory:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB" if torch.cuda.is_available() else "")

import cupy as cp
a = cp.array([1, 2, 3])
print("cupy working:", a.sum())

cuda available: True
gpu name: NVIDIA GeForce RTX 4060 Laptop GPU
gpu memory: 8.59 GB
cupy working: 6


In [3]:
# 3. load expression

import gzip

expr_file = os.path.join(data_dir, "TCGA.BRCA.sampleMap_HiSeqV2_exon.gz")

with gzip.open(expr_file, "rt") as f:
    expr = pd.read_csv(f, sep="\t", index_col=0)

print("expression shape:", expr.shape)
print("first genes:", list(expr.index[:5]))
print("first samples:", list(expr.columns[:3]))

expression shape: (239322, 1218)
first genes: ['chr3:52007981-52008646:-', 'chr1:215901372-215901726:-', 'chr17:49268962-49270377:-', 'chr4:147214081-147214132:-', 'chr1:16074041-16074475:+']
first samples: ['TCGA-A2-A3XU-01', 'TCGA-AR-A2LR-01', 'TCGA-B6-A3ZX-01']


In [4]:
# 4. load copy number

cn_file = os.path.join(data_dir, "TCGA.BRCA.sampleMap_Gistic2_CopyNumber_Gistic2_all_thresholded.by_genes.gz")

with gzip.open(cn_file, "rt") as f:
    cn = pd.read_csv(f, sep="\t", index_col=0)

print("copy number shape:", cn.shape)

copy number shape: (24776, 1080)


In [5]:
# 5. load mutations

mut_file = os.path.join(data_dir, "mc3_BRCA_mc3.txt.gz")

with gzip.open(mut_file, "rt") as f:
    mut = pd.read_csv(f, sep="\t", low_memory=False)

print("mutations shape:", mut.shape)
print("columns:", list(mut.columns[:8]))

mutations shape: (92119, 12)
columns: ['sample', 'chr', 'start', 'end', 'reference', 'alt', 'gene', 'effect']


In [6]:
# 6. load survival

surv_file = os.path.join(data_dir, "BRCA_survival.txt")
surv = pd.read_csv(surv_file, sep="\t")

print("survival shape:", surv.shape)
print("columns:", list(surv.columns))
print(surv.head(3))

survival shape: (1236, 11)
columns: ['sample', '_PATIENT', 'OS', 'OS.time', 'DSS', 'DSS.time', 'DFI', 'DFI.time', 'PFI', 'PFI.time', 'Redaction']
            sample      _PATIENT  OS  OS.time  DSS  DSS.time  DFI  DFI.time  \
0  TCGA-3C-AAAU-01  TCGA-3C-AAAU   0   4047.0  0.0    4047.0  1.0    1808.0   
1  TCGA-3C-AALI-01  TCGA-3C-AALI   0   4005.0  0.0    4005.0  0.0    4005.0   
2  TCGA-3C-AALJ-01  TCGA-3C-AALJ   0   1474.0  0.0    1474.0  0.0    1474.0   

   PFI  PFI.time Redaction  
0    1    1808.0       NaN  
1    0    4005.0       NaN  
2    0    1474.0       NaN  


In [7]:
# 7. load cosmic

cosmic_file = os.path.join(data_dir, "COSMIC_Cancer_Gene_Census.tsv")
cosmic = pd.read_csv(cosmic_file, sep="\t")

print("cosmic shape:", cosmic.shape)
print("columns:", list(cosmic.columns[:6]))

cosmic shape: (763, 20)
columns: ['Gene Symbol', 'Name', 'Entrez GeneId', 'Genome Location', 'Tier', 'Hallmark']


In [8]:
# 8. check sample overlap

expr_samples = set(expr.columns)
cn_samples = set(cn.columns)
surv_samples = set(surv.iloc[:, 0])

overlap = expr_samples & cn_samples & surv_samples

print("expr samples:", len(expr_samples))
print("cn samples:", len(cn_samples))
print("surv samples:", len(surv_samples))
print("three-way overlap:", len(overlap))

expr samples: 1218
cn samples: 1080
surv samples: 1236
three-way overlap: 1077


In [9]:
# 9. inspect gene index format

print("total features:", expr.shape[0])
print("sample index entries:")
for g in list(expr.index[:10]):
    print(" ", g)

# check if any standard gene names exist
symbol_check = [g for g in expr.index if not g.startswith("chr")]
print("non-coordinate entries:", len(symbol_check))
if symbol_check:
    print("examples:", symbol_check[:5])

total features: 239322
sample index entries:
  chr3:52007981-52008646:-
  chr1:215901372-215901726:-
  chr17:49268962-49270377:-
  chr4:147214081-147214132:-
  chr1:16074041-16074475:+
  chr3:195599147-195599341:-
  chr17:30351730-30351801:+
  chr14:106521424-106521433:-
  chr15:42032104-42032401:+
  chr1:94485138-94485315:-
non-coordinate entries: 0


In [10]:
# 10. define target gene panel

hr_genes = [
    "BRCA1", "BRCA2", "PALB2", "RAD51", "RAD51B",
    "RAD51C", "RAD51D", "BRIP1", "ATM", "CHEK2"
]

cohesin_genes = [
    "STAG2", "STAG3", "SMC1A", "SMC1B", "RAD21", "REC8"
]

meiosis_genes = [
    "HORMAD1", "HORMAD2", "SYCP2", "SYCP3",
    "MLH3", "MSH4", "MSH5"
]

receptor_genes = ["ESR1", "PGR", "ERBB2"]

all_panel = hr_genes + cohesin_genes + meiosis_genes
all_genes_needed = all_panel + receptor_genes

print("hr genes:", len(hr_genes))
print("cohesin genes:", len(cohesin_genes))
print("meiosis genes:", len(meiosis_genes))
print("total panel:", len(all_panel))

hr genes: 10
cohesin genes: 6
meiosis genes: 7
total panel: 23


In [11]:
# 11. map coordinates to symbols

# UCSC Xena HiSeqV2 exon uses format gene|coordinate in some versions
# check if pipe separator exists
pipe_check = [g for g in expr.index[:20] if "|" in g]
print("pipe format entries:", len(pipe_check))

# check colon and chromosome pattern
sample_entry = expr.index[0]
print("entry format example:", sample_entry)
parts = sample_entry.replace("chr", "").split(":")
print("split parts:", parts)

pipe format entries: 0
entry format example: chr3:52007981-52008646:-
split parts: ['3', '52007981-52008646', '-']


In [12]:
# 12. load metabric clinical

metabric_clin_file = os.path.join(data_dir, "brca_metabric_clinical_data.tsv")
metabric_clin = pd.read_csv(metabric_clin_file, sep="\t")

print("metabric clinical shape:", metabric_clin.shape)
print("columns:", list(metabric_clin.columns[:10]))

metabric clinical shape: (2509, 39)
columns: ['Study ID', 'Patient ID', 'Sample ID', 'Age at Diagnosis', 'Type of Breast Surgery', 'Cancer Type', 'Cancer Type Detailed', 'Cellularity', 'Chemotherapy', 'Pam50 + Claudin-low subtype']


In [13]:
# 13. check metabric subtype columns

subtype_cols = [c for c in metabric_clin.columns if any(
    x in c.upper() for x in ["SUBTYPE", "PAM50", "ER", "PR", "HER2", "TRIPLE"]
)]
print("subtype related columns:", subtype_cols)
print(metabric_clin[subtype_cols].head(3) if subtype_cols else "none found")

subtype related columns: ['Type of Breast Surgery', 'Cancer Type', 'Cancer Type Detailed', 'Chemotherapy', 'Pam50 + Claudin-low subtype', 'ER status measured by IHC', 'ER Status', 'HER2 status measured by SNP6', 'HER2 Status', 'Tumor Other Histologic Subtype', 'Hormone Therapy', 'Inferred Menopausal State', 'Integrative Cluster', 'Primary Tumor Laterality', 'Nottingham prognostic index', 'Overall Survival (Months)', 'Overall Survival Status', 'PR Status', 'Radio Therapy', 'Number of Samples Per Patient', '3-Gene classifier subtype']
  Type of Breast Surgery    Cancer Type              Cancer Type Detailed  \
0             MASTECTOMY  Breast Cancer  Breast Invasive Ductal Carcinoma   
1      BREAST CONSERVING  Breast Cancer  Breast Invasive Ductal Carcinoma   
2             MASTECTOMY  Breast Cancer  Breast Invasive Ductal Carcinoma   

  Chemotherapy Pam50 + Claudin-low subtype ER status measured by IHC  \
0           NO                 claudin-low                   Positve   
1       

In [14]:
# 14. check cn gene index

print("cn shape:", cn.shape)
print("cn index type examples:")
for g in list(cn.index[:10]):
    print(" ", g)

panel_in_cn = [g for g in all_genes_needed if g in cn.index]
print("panel genes found in cn:", len(panel_in_cn))
print("found:", panel_in_cn)

cn shape: (24776, 1080)
cn index type examples:
  ACAP3
  ACTRT2
  AGRN
  ANKRD65
  ATAD3A
  ATAD3B
  ATAD3C
  AURKAIP1
  B3GALT6
  C1orf159
panel genes found in cn: 26
found: ['BRCA1', 'BRCA2', 'PALB2', 'RAD51', 'RAD51B', 'RAD51C', 'RAD51D', 'BRIP1', 'ATM', 'CHEK2', 'STAG2', 'STAG3', 'SMC1A', 'SMC1B', 'RAD21', 'REC8', 'HORMAD1', 'HORMAD2', 'SYCP2', 'SYCP3', 'MLH3', 'MSH4', 'MSH5', 'ESR1', 'PGR', 'ERBB2']


In [15]:
# 15. check mutation gene column

print("mut columns:", list(mut.columns))
print("mut shape:", mut.shape)

# check which column has gene names
gene_cols = [c for c in mut.columns if any(
    x in c.upper() for x in ["GENE", "HUGO", "SYMBOL"]
)]
print("gene columns:", gene_cols)

if gene_cols:
    sample_genes = mut[gene_cols[0]].unique()
    panel_in_mut = [g for g in all_genes_needed if g in sample_genes]
    print("panel genes found in mut:", len(panel_in_mut))
    print("found:", panel_in_mut)

mut columns: ['sample', 'chr', 'start', 'end', 'reference', 'alt', 'gene', 'effect', 'Amino_Acid_Change', 'DNA_VAF', 'SIFT', 'PolyPhen']
mut shape: (92119, 12)
gene columns: ['gene']
panel genes found in mut: 25
found: ['BRCA1', 'BRCA2', 'PALB2', 'RAD51', 'RAD51B', 'RAD51C', 'RAD51D', 'BRIP1', 'ATM', 'CHEK2', 'STAG2', 'STAG3', 'SMC1A', 'SMC1B', 'RAD21', 'REC8', 'HORMAD1', 'HORMAD2', 'SYCP2', 'MLH3', 'MSH4', 'MSH5', 'ESR1', 'PGR', 'ERBB2']


In [19]:
# 16. query gene coordinates fixed

import mygene
mg = mygene.MyGeneInfo()

query = mg.querymany(
    all_genes_needed,
    scopes="symbol",
    fields="symbol,genomic_pos",
    species="human",
    as_dataframe=True
)

query = query[~query.index.duplicated(keep="first")]

print("columns:", list(query.columns))
print("chr col sample:", query["genomic_pos.chr"].head(3).tolist())
print("start col sample:", query["genomic_pos.start"].head(3).tolist())
print("genes with coords:", query["genomic_pos.chr"].notna().sum())

columns: ['_id', '_score', 'symbol', 'genomic_pos.chr', 'genomic_pos.end', 'genomic_pos.ensemblgene', 'genomic_pos.start', 'genomic_pos.strand', 'genomic_pos']
chr col sample: ['17', '13', '16']
start col sample: [43044292.0, 32315086.0, 23603160.0]
genes with coords: 24


In [20]:
# 17. match exon rows fixed

gene_expr_rows = {}

for gene in all_genes_needed:
    if gene not in query.index:
        print(f"{gene}: not in query")
        continue
    row = query.loc[gene]
    chrom = str(row.get("genomic_pos.chr", ""))
    start = row.get("genomic_pos.start", None)
    end = row.get("genomic_pos.end", None)
    if not chrom or pd.isna(start) or pd.isna(end):
        print(f"{gene}: missing coords")
        continue
    chrom = chrom.strip()
    start = int(start)
    end = int(end)
    matched = []
    for idx in expr.index:
        try:
            parts = idx.split(":")
            c = parts[0].replace("chr", "").strip()
            se = parts[1].split("-")
            s = int(se[0])
            e = int(se[1])
            if c == chrom and not (e < start or s > end):
                matched.append(idx)
        except:
            continue
    gene_expr_rows[gene] = matched
    print(f"{gene}: {len(matched)} exon rows")

BRCA1: 11 exon rows
BRCA2: 17 exon rows
PALB2: 10 exon rows
RAD51: 13 exon rows
RAD51B: 141 exon rows
RAD51C: 5 exon rows
RAD51D: 0 exon rows
BRIP1: 55 exon rows
ATM: 23 exon rows
CHEK2: 3 exon rows
STAG2: 3 exon rows
STAG3: 42 exon rows
SMC1A: 7 exon rows
SMC1B: 4 exon rows
RAD21: 0 exon rows
REC8: missing coords
HORMAD1: 2 exon rows
HORMAD2: 15 exon rows
SYCP2: 0 exon rows
SYCP3: 8 exon rows
MLH3: 5 exon rows
MSH4: 3 exon rows
MSH5: missing coords
ESR1: 40 exon rows
PGR: 3 exon rows
ERBB2: 8 exon rows


In [21]:
# 18. fix missing and zero genes

# manual coords from NCBI GRCh38
manual_coords = {
    "REC8":  {"chr": "14", "start": 23695000,  "end": 23750000},
    "MSH5":  {"chr": "6",  "start": 31924000,  "end": 31943000},
    "RAD51D":{"chr": "17", "start": 33445000,  "end": 33470000},
    "RAD21": {"chr": "8",  "start": 117854000, "end": 117901000},
    "SYCP2": {"chr": "20", "start": 8103000,   "end": 8228000},
}

for gene, coords in manual_coords.items():
    chrom = coords["chr"]
    start = coords["start"]
    end   = coords["end"]
    matched = []
    for idx in expr.index:
        try:
            parts = idx.split(":")
            c = parts[0].replace("chr", "").strip()
            se = parts[1].split("-")
            s = int(se[0])
            e = int(se[1])
            if c == chrom and not (e < start or s > end):
                matched.append(idx)
        except:
            continue
    gene_expr_rows[gene] = matched
    print(f"{gene}: {len(matched)} exon rows after manual fix")

REC8: 1 exon rows after manual fix
MSH5: 36 exon rows after manual fix
RAD51D: 20 exon rows after manual fix
RAD21: 16 exon rows after manual fix
SYCP2: 2 exon rows after manual fix


In [22]:
# 19. aggregate expression per gene

gene_expr_matrix = {}

for gene, rows in gene_expr_rows.items():
    if len(rows) == 0:
        continue
    subset = expr.loc[rows]
    gene_expr_matrix[gene] = subset.mean(axis=0)

gene_expr_df = pd.DataFrame(gene_expr_matrix).T
print("gene expression matrix:", gene_expr_df.shape)
print("genes covered:", list(gene_expr_df.index))

gene expression matrix: (26, 1218)
genes covered: ['BRCA1', 'BRCA2', 'PALB2', 'RAD51', 'RAD51B', 'RAD51C', 'RAD51D', 'BRIP1', 'ATM', 'CHEK2', 'STAG2', 'STAG3', 'SMC1A', 'SMC1B', 'RAD21', 'HORMAD1', 'HORMAD2', 'SYCP2', 'SYCP3', 'MLH3', 'MSH4', 'ESR1', 'PGR', 'ERBB2', 'REC8', 'MSH5']


In [23]:
# 20. log transform and check

gene_expr_df = np.log2(gene_expr_df + 1)

print("expression range min:", round(gene_expr_df.values.min(), 3))
print("expression range max:", round(gene_expr_df.values.max(), 3))
print("any nulls:", gene_expr_df.isnull().sum().sum())

expression range min: 0.0
expression range max: 3.306
any nulls: 0


In [24]:
# 21. save gene expression matrix

out_path = os.path.join(tables_dir, "gene_expression_panel.csv")
gene_expr_df.to_csv(out_path)
print("saved:", out_path)
print("shape:", gene_expr_df.shape)

saved: D:/TNBC_SV_DNA_Repair\results\tables\gene_expression_panel.csv
shape: (26, 1218)


In [25]:
# 22. extract receptor status

esr1  = gene_expr_df.loc["ESR1"]
pgr   = gene_expr_df.loc["PGR"]
erbb2 = gene_expr_df.loc["ERBB2"]

# threshold at median of positive samples
esr1_thresh  = esr1.median()
pgr_thresh   = pgr.median()
erbb2_thresh = erbb2.median()

print("ESR1 median:", round(esr1_thresh, 3))
print("PGR median:", round(pgr_thresh, 3))
print("ERBB2 median:", round(erbb2_thresh, 3))

ESR1 median: 1.882
PGR median: 0.156
ERBB2 median: 0.028


In [26]:
# 23. define TNBC samples

tnbc_mask = (
    (esr1  < esr1_thresh)  &
    (pgr   < pgr_thresh)   &
    (erbb2 < erbb2_thresh)
)

tnbc_samples = esr1.index[tnbc_mask].tolist()
print("total samples:", len(esr1))
print("TNBC samples:", len(tnbc_samples))

total samples: 1218
TNBC samples: 132


In [27]:
# 24. intersect with cn and survival

tnbc_in_cn   = [s for s in tnbc_samples if s in cn.columns]
tnbc_in_surv = [s for s in tnbc_samples if s in surv["sample"].values]
tnbc_final   = list(set(tnbc_in_cn) & set(tnbc_in_surv))

print("TNBC in expression:", len(tnbc_samples))
print("TNBC in copy number:", len(tnbc_in_cn))
print("TNBC in survival:", len(tnbc_in_surv))
print("TNBC final cohort:", len(tnbc_final))

TNBC in expression: 132
TNBC in copy number: 121
TNBC in survival: 132
TNBC final cohort: 121


In [28]:
# 25. subset all modalities

expr_tnbc = gene_expr_df[tnbc_final]
cn_tnbc   = cn.loc[all_panel, tnbc_final]
surv_tnbc = surv[surv["sample"].isin(tnbc_final)].reset_index(drop=True)

print("expression subset:", expr_tnbc.shape)
print("copy number subset:", cn_tnbc.shape)
print("survival subset:", surv_tnbc.shape)

expression subset: (26, 121)
copy number subset: (23, 121)
survival subset: (121, 11)


In [29]:
# 26. subset mutations

mut_tnbc = mut[mut["sample"].isin(tnbc_final)]
print("mutations in TNBC:", mut_tnbc.shape)
print("unique samples in mut:", mut_tnbc["sample"].nunique())

mutations in TNBC: (14338, 12)
unique samples in mut: 90


In [30]:
# 27. save cohort files

expr_tnbc.to_csv(os.path.join(tables_dir, "expr_tnbc.csv"))
cn_tnbc.to_csv(os.path.join(tables_dir, "cn_tnbc.csv"))
surv_tnbc.to_csv(os.path.join(tables_dir, "surv_tnbc.csv"), index=False)
mut_tnbc.to_csv(os.path.join(tables_dir, "mut_tnbc.csv"), index=False)

print("cohort files saved")
print("tables dir:", tables_dir)

cohort files saved
tables dir: D:/TNBC_SV_DNA_Repair\results\tables


In [31]:
# 28. check missing cn genes

missing_cn = [g for g in all_panel if g not in cn.index]
print("genes missing from cn:", missing_cn)

genes missing from cn: []


In [32]:
# 29. annotate panel with cosmic

cosmic_genes = cosmic["Gene Symbol"].values if "Gene Symbol" in cosmic.columns else cosmic.iloc[:, 0].values

panel_cosmic = {}
for gene in all_panel:
    panel_cosmic[gene] = "yes" if gene in cosmic_genes else "no"

cosmic_df = pd.DataFrame.from_dict(
    panel_cosmic, orient="index", columns=["in_cosmic"]
)
cosmic_df["group"] = [
    "HR" if g in hr_genes else
    "cohesin" if g in cohesin_genes else
    "meiosis"
    for g in cosmic_df.index
]

print("panel cosmic annotation:")
print(cosmic_df)
cosmic_df.to_csv(os.path.join(tables_dir, "panel_cosmic_annotation.csv"))

panel cosmic annotation:
        in_cosmic    group
BRCA1         yes       HR
BRCA2         yes       HR
PALB2         yes       HR
RAD51         yes       HR
RAD51B        yes       HR
RAD51C        yes       HR
RAD51D        yes       HR
BRIP1         yes       HR
ATM           yes       HR
CHEK2         yes       HR
STAG2         yes  cohesin
STAG3          no  cohesin
SMC1A         yes  cohesin
SMC1B          no  cohesin
RAD21         yes  cohesin
REC8           no  cohesin
HORMAD1        no  meiosis
HORMAD2        no  meiosis
SYCP2          no  meiosis
SYCP3          no  meiosis
MLH3           no  meiosis
MSH4           no  meiosis
MSH5           no  meiosis


In [33]:
# 30. expression heatmap

matplotlib.use("Agg")

fig, ax = plt.subplots(figsize=(14, 7))

plot_data = expr_tnbc.loc[all_panel].copy()

sns.heatmap(
    plot_data,
    cmap="RdBu_r",
    center=plot_data.values.mean(),
    yticklabels=True,
    xticklabels=False,
    linewidths=0,
    ax=ax,
    cbar_kws={"label": "log2 expression"}
)

ax.set_title("Panel gene expression across TNBC cohort")
ax.set_xlabel("samples (n=121)")
ax.set_ylabel("gene")

plt.tight_layout()
fig.savefig(os.path.join(figures_dir, "nb1_expression_heatmap.png"), dpi=150)
plt.show()
print("saved heatmap")

saved heatmap


In [34]:
# 31. cohort summary

summary = {
    "TNBC samples": len(tnbc_final),
    "panel genes in expression": expr_tnbc.shape[0],
    "panel genes in copy number": cn_tnbc.shape[0],
    "somatic mutations": mut_tnbc.shape[0],
    "mutated samples": mut_tnbc["sample"].nunique(),
    "OS events": int(surv_tnbc["OS"].sum()),
    "median OS days": round(surv_tnbc["OS.time"].median(), 1)
}

for k, v in summary.items():
    print(f"{k}: {v}")

pd.DataFrame.from_dict(
    summary, orient="index", columns=["value"]
).to_csv(os.path.join(tables_dir, "cohort_summary.csv"))
print("notebook 1 complete")

TNBC samples: 121
panel genes in expression: 26
panel genes in copy number: 23
somatic mutations: 14338
mutated samples: 90
OS events: 15
median OS days: 943.0
notebook 1 complete


In [5]:
import os
import pandas as pd

data_dir = 'D:/TNBC_SV_DNA_Repair/dataset'
clin_file = os.path.join(data_dir, 'BRCA_clinicalMatrix.tsv.gz')

clin = pd.read_csv(clin_file, sep='\t', low_memory=False, compression=None)

print('Shape:', clin.shape)
print('First 20 columns:', list(clin.columns[:20]))

pam50_cols = [c for c in clin.columns if 'pam' in c.lower() or 'subtype' in c.lower()]
print('PAM50 related columns:', pam50_cols)

Shape: (1247, 194)
First 20 columns: ['sampleID', 'AJCC_Stage_nature2012', 'Age_at_Initial_Pathologic_Diagnosis_nature2012', 'CN_Clusters_nature2012', 'Converted_Stage_nature2012', 'Days_to_Date_of_Last_Contact_nature2012', 'Days_to_date_of_Death_nature2012', 'ER_Status_nature2012', 'Gender_nature2012', 'HER2_Final_Status_nature2012', 'Integrated_Clusters_no_exp__nature2012', 'Integrated_Clusters_unsup_exp__nature2012', 'Integrated_Clusters_with_PAM50__nature2012', 'Metastasis_Coded_nature2012', 'Metastasis_nature2012', 'Node_Coded_nature2012', 'Node_nature2012', 'OS_Time_nature2012', 'OS_event_nature2012', 'PAM50Call_RNAseq']
PAM50 related columns: ['Integrated_Clusters_with_PAM50__nature2012', 'PAM50Call_RNAseq', 'PAM50_mRNA_nature2012']


In [7]:
import os
import gzip
import pandas as pd

data_dir   = 'D:/TNBC_SV_DNA_Repair/dataset'
tables_dir = 'D:/TNBC_SV_DNA_Repair/results/tables'
scores_dir = 'D:/TNBC_SV_DNA_Repair/results/scores'

hr_genes      = ['BRCA1','BRCA2','PALB2','RAD51','RAD51B','RAD51C','RAD51D','BRIP1','ATM','CHEK2']
cohesin_genes = ['STAG2','STAG3','SMC1A','SMC1B','RAD21','REC8']
meiosis_genes = ['HORMAD1','HORMAD2','SYCP2','SYCP3','MLH3','MSH4','MSH5']
all_panel     = hr_genes + cohesin_genes + meiosis_genes

# Load already-saved cohort data
expr_tnbc = pd.read_csv(os.path.join(tables_dir, 'expr_tnbc.csv'), index_col=0)
cn_tnbc   = pd.read_csv(os.path.join(tables_dir, 'cn_tnbc.csv'), index_col=0)
surv_tnbc = pd.read_csv(os.path.join(tables_dir, 'surv_tnbc.csv'))
mut_tnbc  = pd.read_csv(os.path.join(tables_dir, 'mut_tnbc.csv'))
tnbc_final = list(expr_tnbc.columns)

# Load full expression and CN for subsetting
with gzip.open(os.path.join(data_dir, 'TCGA.BRCA.sampleMap_HiSeqV2_exon.gz'), 'rt') as f:
    expr_full = pd.read_csv(f, sep='\t', index_col=0, nrows=0)  # header only
expr_samples = set(expr_full.columns)

with gzip.open(os.path.join(data_dir, 'TCGA.BRCA.sampleMap_Gistic2_CopyNumber_Gistic2_all_thresholded.by_genes.gz'), 'rt') as f:
    cn_full = pd.read_csv(f, sep='\t', index_col=0, nrows=0)
cn_samples = set(cn_full.columns)

surv_full    = pd.read_csv(os.path.join(data_dir, 'BRCA_survival.txt'), sep='\t')
surv_samples = set(surv_full['sample'])

# PAM50 cohort — column uses 'Basal' not 'Basal-like'
pam50_basal = set(clin[clin['PAM50Call_RNAseq'] == 'Basal']['sampleID'].tolist())
print(f'PAM50 Basal samples: {len(pam50_basal)}')

pam50_tnbc_final = sorted(pam50_basal & expr_samples & cn_samples & surv_samples)
print(f'PAM50 TNBC with complete data: {len(pam50_tnbc_final)}')

overlap = len(set(pam50_tnbc_final) & set(tnbc_final))
print(f'\nOverlap with expression-threshold cohort: {overlap}')
print(f'In PAM50 only: {len(set(pam50_tnbc_final) - set(tnbc_final))}')
print(f'In expression-threshold only: {len(set(tnbc_final) - set(pam50_tnbc_final))}')
print(f'\nExpression-threshold cohort size: {len(tnbc_final)}')
print(f'PAM50 cohort size: {len(pam50_tnbc_final)}')

PAM50 Basal samples: 142
PAM50 TNBC with complete data: 135

Overlap with expression-threshold cohort: 20
In PAM50 only: 115
In expression-threshold only: 101

Expression-threshold cohort size: 121
PAM50 cohort size: 135


In [8]:
# Check for clinical receptor status columns
receptor_cols = [c for c in clin.columns if any(x in c.lower() 
                 for x in ['er_status', 'pr_status', 'her2', 'receptor'])]
print('Receptor status columns:', receptor_cols)
print(clin[receptor_cols].head(5))

Receptor status columns: ['ER_Status_nature2012', 'HER2_Final_Status_nature2012', 'PR_Status_nature2012', 'breast_carcinoma_estrogen_receptor_status', 'breast_carcinoma_progesterone_receptor_status', 'her2_and_centromere_17_positive_finding_other_measuremnt_scl_txt', 'her2_erbb_method_calculation_method_text', 'her2_erbb_pos_finding_cell_percent_category', 'her2_erbb_pos_finding_fluorescence_n_st_hybrdztn_clcltn_mthd_txt', 'her2_immunohistochemistry_level_result', 'her2_neu_and_centromere_17_copy_number_analysis_npt_ttl_nmbr_cnt', 'her2_neu_breast_carcinoma_copy_analysis_input_total_number', 'her2_neu_chromosone_17_signal_ratio_value', 'her2_neu_metastatic_breast_carcinoma_copy_analysis_inpt_ttl_nmbr', 'lab_proc_her2_neu_immunohistochemistry_receptor_status', 'lab_procedure_her2_neu_in_situ_hybrid_outcome_type', 'metastatic_breast_carcinoma_estrogen_receptor_detection_mthd_txt', 'metastatic_breast_carcinoma_estrogen_receptor_status', 'metastatic_breast_carcinoma_her2_erbb_method_calcul

In [9]:
# Check value distributions and coverage
er_col   = 'breast_carcinoma_estrogen_receptor_status'
pr_col   = 'breast_carcinoma_progesterone_receptor_status'
her2_col = 'lab_proc_her2_neu_immunohistochemistry_receptor_status'

print('ER values:', clin[er_col].value_counts().to_dict())
print('PR values:', clin[pr_col].value_counts().to_dict())
print('HER2 values:', clin[her2_col].value_counts().to_dict())

# How many samples have all three
complete = clin[[er_col, pr_col, her2_col, 'sampleID']].dropna()
print(f'\nSamples with all three receptor annotations: {len(complete)}')

# Define clinical TNBC
clinical_tnbc = clin[
    (clin[er_col]   == 'Negative') &
    (clin[pr_col]   == 'Negative') &
    (clin[her2_col] == 'Negative')
]['sampleID'].tolist()

print(f'Clinical TNBC (all three Negative): {len(clinical_tnbc)}')

# Overlap with our cohort
clinical_tnbc_set = set(clinical_tnbc)
overlap_clinical = len(clinical_tnbc_set & set(tnbc_final))
print(f'Overlap with expression-threshold cohort: {overlap_clinical}')
print(f'Overlap with PAM50 Basal cohort: {len(clinical_tnbc_set & set(pam50_tnbc_final))}')

ER values: {'Positive': 912, 'Negative': 266, 'Indeterminate': 3}
PR values: {'Positive': 791, 'Negative': 383, 'Indeterminate': 5}
HER2 values: {'Negative': 641, 'Equivocal': 195, 'Positive': 187, 'Indeterminate': 12}

Samples with all three receptor annotations: 1032
Clinical TNBC (all three Negative): 130
Overlap with expression-threshold cohort: 22
Overlap with PAM50 Basal cohort: 73


In [10]:
# Build clinical TNBC cohort
clin_tnbc_with_data = sorted(
    clinical_tnbc_set & expr_samples & cn_samples & surv_samples
)
print(f'Clinical TNBC with complete data: {len(clin_tnbc_with_data)}')

# Load full expression for panel genes
gene_expr_df = pd.read_csv(os.path.join(tables_dir, 'gene_expression_panel.csv'),
                           index_col=0)

all_panel = hr_genes + cohesin_genes + meiosis_genes

expr_clin = gene_expr_df[[s for s in clin_tnbc_with_data
                           if s in gene_expr_df.columns]]

with gzip.open(os.path.join(data_dir,
    'TCGA.BRCA.sampleMap_Gistic2_CopyNumber_Gistic2_all_thresholded.by_genes.gz'),
    'rt') as f:
    cn_full_df = pd.read_csv(f, sep='\t', index_col=0)

cn_clin  = cn_full_df.loc[all_panel,
           [s for s in clin_tnbc_with_data if s in cn_full_df.columns]]

surv_clin = surv_full[surv_full['sample'].isin(clin_tnbc_with_data)].reset_index(drop=True)

with gzip.open(os.path.join(data_dir, 'mc3_BRCA_mc3.txt.gz'), 'rt') as f:
    mut_full = pd.read_csv(f, sep='\t', low_memory=False)
mut_clin = mut_full[mut_full['sample'].isin(clin_tnbc_with_data)]

# Final intersection
final_clin = sorted(
    set(expr_clin.columns) & set(cn_clin.columns) & set(surv_clin['sample'])
)
print(f'Final clinical cohort size: {len(final_clin)}')
print(f'OS events: {int(surv_clin[surv_clin["sample"].isin(final_clin)]["OS"].sum())}')
print(f'PFI events: {int(surv_clin[surv_clin["sample"].isin(final_clin)]["PFI"].sum())}')

# Save
expr_clin[final_clin].to_csv(os.path.join(tables_dir, 'expr_tnbc_clinical.csv'))
cn_clin[final_clin].to_csv(os.path.join(tables_dir, 'cn_tnbc_clinical.csv'))
surv_clin[surv_clin['sample'].isin(final_clin)].to_csv(
    os.path.join(tables_dir, 'surv_tnbc_clinical.csv'), index=False)
mut_clin.to_csv(os.path.join(tables_dir, 'mut_tnbc_clinical.csv'), index=False)

print('Clinical cohort files saved')

Clinical TNBC with complete data: 112
Final clinical cohort size: 112
OS events: 18
PFI events: 18
Clinical cohort files saved
